# STGAN ConvGRU su ERA5: il modello contro i rilevatori senza modello

Run `stgan_convgru_fullgrid_recent16_cyclic_bf16_cleanD_wdaily_seasonal_trend_ep3`: 3 epoche, training
1980–2004, test 2005–2025. Ha tutte le aggiunte: griglia intera, storia di 16 passi, fix del dropout del
discriminatore, normalizzazione stagionale con trend, pesi delle variabili `daily_change`, MC Dropout con 20
campioni.

Quattro **rilevatori**, ognuno con lo stesso numero di segnalazioni:

| Rilevatore | Cosa misura | Modello? |
|---|---|---|
| **modello** | errore di ricostruzione dello STGAN | sì |
| **distanza dal normale** | quanto il valore si allontana dal normale stagionale di cella, giorno e ora | no |
| **variazione in 3 h** | quanto il valore è cambiato rispetto all'istante prima | no |
| **normale + variazione** | il più alto dei due precedenti (sono sulla stessa scala) | no |

Tutti sono letti dopo la climatologia, cioè standardizzati per cella, mese e ora. Un valore (cella × istante)
è **segnalato** da un rilevatore se è nel suo 5% più alto (tutte le segnalazioni) o nel suo 1% più alto (solo
le più forti). Siccome ognuno segnala la stessa quota, il confronto è alla pari.

Il notebook legge solo i file della run, senza GPU, su un istante ogni `STRIDE`. Le quote possono quindi
differire di poco da quelle dei log completi.

**Avvertenze.** Un solo seed. Climatologia e soglie sono stimate sugli anni di test: nella versione finale
saranno stimate sulla validazione.

In [ ]:
from pathlib import Path
import json, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
from IPython.display import display

ROOT = Path(os.environ['PHYSIQ_ROOT']) if os.environ.get('PHYSIQ_ROOT') else next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'physiq_pv').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Apri il notebook dalla cartella del repository physiq_pv')
OUT = ROOT / 'outputs'
RUN = OUT / 'stgan_convgru_fullgrid_recent16_cyclic_bf16_cleanD_wdaily_seasonal_trend_ep3'
SCORES = {  # stesso periodo e stesse celle, tutti dopo la climatologia
    'modello': Path(f'{RUN}_gen_clim'),
    'distanza dal normale': Path(f'{RUN}_no_model_bal_clim'),
    'variazione in 3 h': Path(f'{RUN}_previous_bal_clim'),
}
MODEL, NORMAL, CHANGE = SCORES
COMBINED = 'normale + variazione'
DETECTORS = [MODEL, NORMAL, CHANGE, COMBINED]
OTHERS = DETECTORS[1:]
OTHER = COMBINED      # rilevatore del confronto dettagliato (sezioni 2 e 3): anche NORMAL o CHANGE
LEVELS = (5, 1)       # segnalato = nel 5% / nell'1% più alto di ciascun rilevatore
LEAD_LOGS = ['anticipo_modello_tutti.log', 'anticipo_controlli.log', 'anticipo_modello_precedente_forti.log']
STRIDE = 29           # un istante ogni 29 (dispari: si leggono tutte le ore del giorno)
MIN_CELLS = 10        # perturbazioni da almeno 10 celle (circa 25.000 km²)
CONTROL_DAYS = 30     # controllo del caso: segnalazioni del modello di circa un mese dopo
SAVE_FIGURES = False
FIGURE_DIR = OUT / 'figure_ep3'
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25})

def show(figure, name):
    if SAVE_FIGURES:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        figure.savefig(FIGURE_DIR / f'{name}.png', bbox_inches='tight', dpi=150)
    plt.show()

def manifest(directory):
    return json.loads((directory / 'metadata.json').read_text(encoding='utf-8'))

def open_scores(directory, spread=False):
    data = manifest(directory)
    file = (data.get('anomaly_std_file') if spread
            else data.get('anomaly_mean_file', data.get('scores_file', 'scores/anomaly_mean.npy')))
    return None if file is None or not (directory / file).is_file() else np.load(directory / file, mmap_mode='r')

def find_log(name):
    return next((path for path in (ROOT / name, OUT / name) if path.is_file()), None)

def percent(value):
    return f'{value:.1%}'

## 0. La run

Configurazione letta dall'ultimo checkpoint.

In [ ]:
checkpoints = sorted(RUN.glob('model_epoch_*.pt'), key=lambda path: int(re.search(r'_epoch_(\d+)', path.name).group(1)))
if checkpoints:
    import torch
    payload = torch.load(checkpoints[-1], map_location='cpu', weights_only=False)
    rates, model_config = payload.get('learning_rates') or {}, payload.get('model_config') or {}
    display(pd.Series({
        'epoche': payload.get('completed_epochs'), 'training': (payload.get('splits') or {}).get('train'),
        'lr G': rates.get('generator_learning_rate'), 'lr D': rates.get('discriminator_learning_rate'),
        'passi D:G': payload.get('discriminator_generator_update_ratio', '1:1'),
        'peso ricostruzione': payload.get('generator_reconstruction_weight'),
        'normalizzazione': (payload.get('normalization') or {}).get('kind'),
        'pesi delle variabili': 'sì' if model_config.get('reconstruction_feature_weights') is not None else 'no',
        'D vede la ricostruzione senza dropout': model_config.get('discriminator_input_dropout') is False,
        'campioni MC': (payload.get('mc_config') or {}).get('mc_samples'),
    }, name='valore').to_frame())
    del payload
else:
    print('Checkpoint non trovati in', RUN)

In [ ]:
# Lettura: ogni istante letto insieme a quello dopo (per la persistenza).
run_metadata = manifest(RUN)
grid = run_metadata['grid']
times_all = pd.to_datetime(np.load(SCORES[MODEL] / 'test_timestamps.npy'))
rows_read = np.arange(0, len(times_all) - 1, STRIDE)
times = times_all[rows_read]
value, following = {}, {}
for name, directory in SCORES.items():
    if not pd.to_datetime(np.load(directory / 'test_timestamps.npy')).equals(times_all):
        raise ValueError(f'{name}: non è sugli stessi istanti del modello')
    scores = open_scores(directory)
    value[name] = np.asarray(scores[rows_read], dtype=np.float32)
    following[name] = np.asarray(scores[rows_read + 1], dtype=np.float32)
    del scores
finite = np.logical_and.reduce([np.isfinite(values) for values in value.values()])
value[COMBINED] = np.fmax(value[NORMAL], value[CHANGE])
following[COMBINED] = np.fmax(following[NORMAL], following[CHANGE])
limit = {level: {name: float(np.percentile(value[name][finite], 100 - level)) for name in DETECTORS} for level in LEVELS}
flag = {level: {name: (value[name] > limit[level][name]) & finite for name in DETECTORS} for level in LEVELS}
flag_next = {level: {name: following[name] > limit[level][name] for name in DETECTORS} for level in LEVELS}
latitudes, longitudes = np.asarray(grid['latitudes']), np.asarray(grid['longitudes'])
cell_rows, cell_cols = np.asarray(grid['rows']), np.asarray(grid['cols'])
cell_latitude = latitudes[cell_rows]
print(f'{len(times)} istanti letti, dal {times[0].date()} al {times[-1].date()}; {finite.shape[1]} celle')
display(pd.DataFrame(limit).rename(columns=lambda level: f'soglia {level}%').round(3))

## 1. Quanto il modello si sovrappone agli altri

Ogni casella: dei valori segnalati dal rilevatore della **riga**, la quota segnalata anche da quello della
**colonna**. Per la riga del modello, quello che manca al 100% è ciò che trova solo il modello rispetto a
quel rilevatore.

In [ ]:
figure, axes = plt.subplots(1, len(LEVELS), figsize=(6.5 * len(LEVELS), 5), constrained_layout=True)
overlap = {}
for axis, level in zip(np.atleast_1d(axes), LEVELS):
    matrix = np.array([[(flag[level][a] & flag[level][b]).sum() / flag[level][a].sum() for b in DETECTORS]
                       for a in DETECTORS])
    overlap[level] = pd.DataFrame(matrix, index=DETECTORS, columns=DETECTORS)
    axis.imshow(100 * matrix, cmap='Blues', vmin=0, vmax=100)
    for i in range(len(DETECTORS)):
        for j in range(len(DETECTORS)):
            axis.text(j, i, f'{100 * matrix[i, j]:.0f}%', ha='center', va='center',
                      color='white' if matrix[i, j] > 0.6 else 'black')
    axis.set_xticks(range(len(DETECTORS)), DETECTORS, rotation=25, ha='right')
    axis.set_yticks(range(len(DETECTORS)), DETECTORS)
    axis.set_title(f'Segnalati al {level}%: quota anche della colonna'); axis.grid(False)
show(figure, 'sovrapposizione')
only_model = pd.DataFrame({f'{level}%': {other: 1 - overlap[level].loc[MODEL, other] for other in OTHERS}
                           for level in LEVELS})
print("Dei valori segnalati dal modello, quota che l'altro rilevatore NON segnala:")
display(only_model.style.format('{:.1%}'))

## 2. Che tipi di evento sono

Confronto tra il modello e il rilevatore `OTHER` (per default **normale + variazione**; si può cambiare nella
prima cella). I valori sono divisi in quattro gruppi:

| Gruppo | Modello | `OTHER` |
|---|---|---|
| **solo modello** | ✔ | ✘ |
| **entrambi** | ✔ | ✔ |
| **solo `OTHER`** | ✘ | ✔ |
| **nessuno** (riferimento) | ✘ | ✘ |

Per ogni gruppo:
- il percentile mediano in ciascun rilevatore. Un valore al 50° percentile della distanza dal normale non è
  lontano dal normale; uno al 99° lo è molto;
- quanti sono segnalati di nuovo 3 ore dopo, nella stessa cella;
- la stagione e la latitudine;
- di cosa è fatto l'errore del modello, cioè la quota di ogni variabile nell'errore pesato.

In [ ]:
def groups(level, other=OTHER):
    model, rival = flag[level][MODEL], flag[level][other]
    return {'solo modello': model & ~rival, 'entrambi': model & rival, f'solo {other}': ~model & rival,
            'nessuno': ~model & ~rival & finite}

GROUP_COLORS = ['tab:red', 'tab:purple', 'tab:blue', 'tab:grey']
SEASONS = ('inverno', 'primavera', 'estate', 'autunno')
season_of = np.array(['inverno', 'inverno', 'primavera', 'primavera', 'primavera', 'estate', 'estate', 'estate',
                      'autunno', 'autunno', 'autunno', 'inverno'])[times.month.to_numpy() - 1]
reference = {name: np.sort(value[name][finite]) for name in DETECTORS}
def percentile_of(name, mask):
    return 100 * np.searchsorted(reference[name], value[name][mask]) / len(reference[name])

group_tables = {}
for level in LEVELS:
    rows = []
    for group, mask in groups(level).items():
        count = mask.sum()
        rows.append({'gruppo': group, '% di tutti i valori': count / finite.sum(),
                     **{f'percentile in {name}': np.median(percentile_of(name, mask)) for name in DETECTORS},
                     'di nuovo dal modello 3 h dopo': (flag_next[level][MODEL] & mask).sum() / count,
                     f'di nuovo da {OTHER} 3 h dopo': (flag_next[level][OTHER] & mask).sum() / count,
                     'latitudine mediana': float(np.median(np.broadcast_to(cell_latitude, mask.shape)[mask])),
                     **{season: (mask & (season_of == season)[:, None]).sum() / count for season in SEASONS}})
    group_tables[level] = pd.DataFrame(rows).set_index('gruppo')
    print(f'Segnalati al {level}%, modello contro {OTHER}')
    table = group_tables[level]
    display(table.style.format({column: '{:.1%}' for column in table if column.startswith(('%', 'di nuovo'))
                                or column in SEASONS}
                               | {column: '{:.0f}' for column in table if column.startswith('percentile')}
                               | {'latitudine mediana': '{:.1f}'}))

figure, axes = plt.subplots(len(LEVELS), 2, figsize=(14, 3.8 * len(LEVELS)), constrained_layout=True, squeeze=False)
for row, level in enumerate(LEVELS):
    table = group_tables[level]
    (table[[f'percentile in {name}' for name in DETECTORS]].rename(columns=lambda c: c[len('percentile in '):])
     .plot.barh(ax=axes[row, 0], width=0.8))
    axes[row, 0].axvline(100 - level, color='grey', linestyle=':')
    axes[row, 0].set(xlabel='percentile mediano', title=f'{level}%: dove stanno i gruppi in ciascun rilevatore')
    axes[row, 0].invert_yaxis(); axes[row, 0].legend(fontsize=7)
    (100 * table[list(SEASONS)]).plot.barh(stacked=True, ax=axes[row, 1], colormap='coolwarm')
    axes[row, 1].set(xlabel='% del gruppo', title=f'{level}%: stagione'); axes[row, 1].invert_yaxis()
    axes[row, 1].legend(fontsize=7, loc='lower right')
show(figure, 'tipi_di_evento')

In [ ]:
feature_file = RUN / 'scores' / 'feature_scores.npy'
weights = (run_metadata.get('backend') or {}).get('reconstruction_feature_weights')
names = (run_metadata.get('preparation') or {}).get('features') or (list(weights) if weights else None)
if feature_file.is_file() and names:
    errors = np.load(feature_file, mmap_mode='r')
    weight = np.array([weights[name] for name in names]) if weights else np.ones(len(names))
    masks = {level: groups(level) for level in LEVELS}
    sums = {level: {group: np.zeros(len(names)) for group in masks[level]} for level in LEVELS}
    for start in range(0, len(rows_read), 8):
        block = np.asarray(errors[rows_read[start:start + 8]], dtype=np.float64) * weight
        usable = np.isfinite(block).all(axis=-1)
        for level in LEVELS:
            for group, mask in masks[level].items():
                sums[level][group] += block[mask[start:start + 8] & usable].sum(axis=0)
    del errors
    figure, axes = plt.subplots(1, len(LEVELS), figsize=(8 * len(LEVELS), 3.8), constrained_layout=True, squeeze=False)
    for axis, level in zip(axes[0], LEVELS):
        composition = pd.DataFrame({group: total / total.sum() for group, total in sums[level].items()}, index=names).T
        order = composition.loc['solo modello'].sort_values(ascending=False).index
        shown = composition[list(order[:7])].assign(altre=composition[list(order[7:])].sum(axis=1))
        (100 * shown).plot.barh(stacked=True, ax=axis, colormap='tab10', legend=level == LEVELS[-1])
        axis.set(xlabel="% dell'errore pesato del modello", title=f"{level}%: di cosa è fatto l'errore del modello")
        axis.invert_yaxis()
        print(f'Segnalati al {level}%')
        display(composition[order].style.format('{:.1%}'))
    axes[0, -1].legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left')
    show(figure, 'errore_per_variabile')
else:
    print('Errori per variabile non disponibili:', feature_file)

## 3. L'incertezza MC Dropout

Lo score del modello è la media dei 20 campioni MC Dropout; lo **spread** è la loro deviazione standard, nelle
stesse unità dello score. Una segnalazione del modello è **solida** se resta sopra la soglia anche abbassando
lo score di uno spread: in quel caso non dipende dal dropout.

La tabella è per il confronto con `OTHER`. Il riquadro sotto riporta le segnalazioni solide del modello
contro ciascuno dei tre rilevatori.

In [ ]:
spreads = open_scores(SCORES[MODEL], spread=True)
if spreads is not None:
    spread = np.asarray(spreads[rows_read], dtype=np.float32)
    del spreads
    usable = finite & np.isfinite(spread)
    def solid(level, mask):
        chosen = mask & usable
        return float(((value[MODEL] - spread) > limit[level][MODEL])[chosen].mean()) if chosen.any() else np.nan
    dropout_tables = {}
    for level in LEVELS:
        rows = []
        for group, mask in groups(level).items():
            row = {'gruppo': group, 'spread mediano': float(np.median(spread[mask & usable]))}
            if group in ('solo modello', 'entrambi'):
                chosen = mask & usable
                row['segnalazioni solide'] = solid(level, mask)
                row['spread / distanza dalla soglia (mediana)'] = float(
                    np.median(spread[chosen] / (value[MODEL][chosen] - limit[level][MODEL])))
            rows.append(row)
        dropout_tables[level] = pd.DataFrame(rows).set_index('gruppo')
        print(f'Segnalati al {level}%, modello contro {OTHER}')
        display(dropout_tables[level].style.format({'spread mediano': '{:.3f}', 'segnalazioni solide': '{:.1%}',
                                                    'spread / distanza dalla soglia (mediana)': '{:.2f}'}, na_rep='—'))
    solid_by_other = pd.DataFrame({(f'{level}%', group): {
        other: solid(level, groups(level, other)[group]) for other in OTHERS}
        for level in LEVELS for group in ('solo modello', 'entrambi')})
    print('Segnalazioni solide del modello, contro ciascun rilevatore')
    display(solid_by_other.style.format('{:.1%}'))

    figure, axes = plt.subplots(1, len(LEVELS) + 1, figsize=(5.5 * (len(LEVELS) + 1), 3.8), constrained_layout=True)
    for axis, level in zip(axes, LEVELS):
        table = dropout_tables[level]
        axis.barh(table.index[::-1], table['spread mediano'][::-1], color=GROUP_COLORS[::-1])
        axis.set(title=f'{level}%: spread mediano dei campioni MC', xlabel='deviazione standard dello score')
    sample = np.random.default_rng(0).choice(np.flatnonzero(usable.ravel()), size=min(400_000, int(usable.sum())), replace=False)
    shown = axes[-1].hexbin(value[MODEL].ravel()[sample], spread.ravel()[sample], gridsize=80, bins='log', cmap='viridis')
    for level, style in zip(LEVELS, ('--', ':')):
        axes[-1].axvline(limit[level][MODEL], color='red', linestyle=style, label=f'soglia {level}%')
    axes[-1].set(xlabel='score del modello', ylabel='spread MC', title='Score e spread'); axes[-1].legend()
    figure.colorbar(shown, ax=axes[-1], label='valori (log)')
    show(figure, 'mc_dropout')
else:
    print('La run della climatologia non ha lo spread dei campioni MC')

## 4. Di quanto il modello allarga le perturbazioni degli altri rilevatori

A ogni istante, le celle segnalate da un rilevatore senza modello formano delle **perturbazioni**: gruppi di
celle vicine, contando anche le diagonali. Si aggiungono le celle segnalate dal modello allo stesso livello.
Una perturbazione si allarga se il modello segnala celle attaccate a essa.

**Allargamento** = area finale / area di partenza: 1 vuol dire nessuna aggiunta, 2 vuol dire area doppia. Le
aree sono in km², con il coseno della latitudine. Si contano le perturbazioni di almeno `MIN_CELLS` celle.

**Controllo del caso**: lo stesso calcolo con le segnalazioni del modello di circa `CONTROL_DAYS` giorni dopo.
L'allargamento vero del modello è quello che supera il controllo.

In [ ]:
height, width = len(latitudes), len(longitudes)
cell_km2 = (111.32 * float(grid.get('spacing', 0.5))) ** 2 * np.cos(np.deg2rad(cell_latitude))
STRUCTURE = np.ones((3, 3), dtype=bool)
shift = max(1, round(CONTROL_DAYS * 24 / (3 * STRIDE)))  # istanti letti in circa CONTROL_DAYS giorni

def perturbations(model_flags, base_flags):
    pieces = []
    for step in range(len(base_flags)):
        base, model, area = np.zeros((height, width), bool), np.zeros((height, width), bool), np.zeros((height, width))
        base[cell_rows, cell_cols], model[cell_rows, cell_cols], area[cell_rows, cell_cols] = (
            base_flags[step], model_flags[step], cell_km2)
        labels, count = ndimage.label(base | model, structure=STRUCTURE)
        if not count:
            continue
        labels = labels.ravel()
        total = lambda weights: np.bincount(labels, weights=weights.ravel(), minlength=count + 1)[1:]
        pieces.append(np.column_stack((total(base.astype(float)), total(base * area), total((base | model) * area),
                                       total((model & ~base) * area))))
    return pd.DataFrame(np.vstack(pieces), columns=['celle di partenza', 'area di partenza', 'area finale',
                                                    'area solo modello'])

def widening(table):
    big = table[table['celle di partenza'] >= MIN_CELLS]
    ratio = big['area finale'] / big['area di partenza']
    added = table['area solo modello']
    return big, ratio, {
        'perturbazioni': len(big),
        'area mediana di partenza (km²)': big['area di partenza'].median(),
        'allargamento mediano': ratio.median(),
        'allargamento complessivo (somma delle aree)': big['area finale'].sum() / big['area di partenza'].sum(),
        'allargate almeno del 50%': (ratio >= 1.5).mean(),
        'area solo modello attaccata a una perturbazione': added[table['celle di partenza'] > 0].sum() / max(added.sum(), 1),
    }

area_results, area_summary = {}, {}
for level in LEVELS:
    model_flags = flag[level][MODEL]
    for other in OTHERS:
        for label, flags in (('modello', model_flags), ('caso', np.roll(model_flags, -shift, axis=0))):
            big, ratio, numbers = widening(perturbations(flags, flag[level][other]))
            area_results[level, other, label] = (big, ratio)
            area_summary[(f'{level}%', other, label)] = numbers
area_summary = pd.DataFrame(area_summary)
for level in LEVELS:
    print(f'Segnalati al {level}%: perturbazioni di ciascun rilevatore, allargate dal modello e dal caso')
    display(area_summary[f'{level}%'].style.format('{:,.2f}'))

figure, axes = plt.subplots(len(LEVELS), 2, figsize=(14, 3.8 * len(LEVELS)), constrained_layout=True, squeeze=False)
bins = np.geomspace(1, 20, 40)
size_bins = [MIN_CELLS, 30, 100, 300, np.inf]
size_labels = [f'{a:.0f}–{b:.0f}' if np.isfinite(b) else f'≥{a:.0f}' for a, b in zip(size_bins[:-1], size_bins[1:])]
for row, level in enumerate(LEVELS):
    for (label, color), offset in zip((('modello', 'tab:red'), ('caso', 'tab:grey')), (-0.2, 0.2)):
        big, ratio = area_results[level, OTHER, label]
        axes[row, 0].hist(ratio.clip(upper=20), bins=bins, alpha=0.6, color=color, label=label)
        by_size = ratio.groupby(pd.cut(big['celle di partenza'], size_bins, right=False), observed=False).median()
        axes[row, 1].bar(np.arange(len(by_size)) + offset, by_size.values, width=0.4, color=color, label=label)
    axes[row, 0].set(xscale='log', xlabel='allargamento (area finale / area di partenza)', ylabel='perturbazioni',
                     title=f'{level}%: perturbazioni di {OTHER}'); axes[row, 0].legend()
    axes[row, 1].set_xticks(range(len(size_labels)), size_labels)
    axes[row, 1].axhline(1, color='black', linewidth=0.8)
    axes[row, 1].set(xlabel='celle della perturbazione di partenza', ylabel='allargamento mediano',
                     title=f'{level}%: per dimensione'); axes[row, 1].legend()
show(figure, 'allargamento_aree')

## 5. Di quanto anticipa

Tabelle di `scripts/check_era5_lead.py`, calcolate su tutti gli istanti e le celle (log `anticipo_*.log`).
Per ogni inizio di un episodio del rilevatore della riga, nella stessa cella, si guarda se l'altro rilevatore
era **già acceso**, e da quante ore. Le fasce colorate dividono il "già acceso" per numero di ore. Il pallino
nero è il caso: la stessa misura in istanti in cui il rilevatore della riga è quieto e non comincia nulla. I
nomi sono quelli delle cartelle: `gen_clim` = modello, `no_model_bal_clim` = distanza dal normale,
`previous_bal_clim` = variazione in 3 h, `simple_max` = normale + variazione.

In [ ]:
COLUMNS = ['starts', 'already', '3h', '6h', '9-12h', '15-24h', '>24h', 'lead med h', 'gap', 'together',
           'later', 'lag med h', 'none']
BANDS = ['3h', '6h', '9-12h', '15-24h', '>24h']
blocks = []
for name in LEAD_LOGS:
    path = find_log(name)
    if path is None:
        print('Log non trovato:', name)
        continue
    for line in path.read_text(encoding='utf-8', errors='replace').splitlines():
        if line.startswith('[lead]'):
            blocks.append({'log': name, 'levels': line.split('|')[1].strip(), 'rows': {}})
        elif blocks and line.startswith(('starts of', '  chance')):
            values = line[34:].split()
            if len(values) == len(COLUMNS):
                blocks[-1]['rows'][line[:34].strip()] = [float(value) for value in values]
bars = []
for block in blocks:
    table = pd.DataFrame(block['rows'], index=COLUMNS).T
    print(f"{block['log']} | {block['levels']}")
    display(table.round(1))
    labels = list(table.index)
    for index, label in enumerate(labels):
        if label.startswith('starts of'):
            chance = labels[index + 1] if index + 1 < len(labels) and labels[index + 1].startswith('chance') else None
            bars.append({'riga': f"{label} ({block['levels'].replace('flagged: the highest ', '')})",
                         **table.loc[label, BANDS].to_dict(),
                         'caso': table.loc[chance, 'already'] if chance else np.nan})
if bars:
    bars = pd.DataFrame(bars).set_index('riga')
    figure, axis = plt.subplots(figsize=(11, 0.45 * len(bars) + 1.5), constrained_layout=True)
    bars[BANDS].plot.barh(stacked=True, ax=axis, colormap='YlOrRd', width=0.7)
    axis.scatter(bars['caso'], np.arange(len(bars)), color='black', zorder=3, label='caso')
    axis.set(xlabel="% degli inizi di episodio in cui l'altro rilevatore era già acceso", ylabel='')
    axis.set_yticks(range(len(bars)), bars.index, fontsize=7)
    axis.invert_yaxis(); axis.legend(title='acceso da', fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
    show(figure, 'anticipo')

## 6. In sintesi

Per ciascun rilevatore senza modello e per i due livelli, i numeri principali calcolati sopra.

In [ ]:
recap = {}
for level in LEVELS:
    for other in OTHERS:
        column = {'segnalati dal modello che l\'altro non segnala': percent(1 - overlap[level].loc[MODEL, other]),
                  'segnalati dall\'altro che il modello non segnala': percent(1 - overlap[level].loc[other, MODEL])}
        if 'solid_by_other' in globals():
            column['solide, solo modello'] = percent(solid_by_other.loc[other, (f'{level}%', 'solo modello')])
            column['solide, entrambi'] = percent(solid_by_other.loc[other, (f'{level}%', 'entrambi')])
        column['allargamento complessivo, modello'] = (
            f"{area_summary.loc['allargamento complessivo (somma delle aree)', (f'{level}%', other, 'modello')]:.2f}")
        column['allargamento complessivo, caso'] = (
            f"{area_summary.loc['allargamento complessivo (somma delle aree)', (f'{level}%', other, 'caso')]:.2f}")
        recap[(f'{level}%', other)] = column
display(pd.DataFrame(recap))

**Come leggerli**
- Il confronto è alla pari: ogni rilevatore segnala la stessa quota di valori.
- Quello che segnala solo il modello va guardato nella sezione 2: dove sta negli altri rilevatori, quanto
  dura, di cosa è fatto.
- L'allargamento delle aree conta solo se supera quello del controllo del caso.
- L'anticipo è rispetto agli altri rilevatori, non una previsione: ERA5 arriva con qualche giorno di ritardo.
- Un solo seed, e soglie e climatologia stimate sugli anni di test.